# Nifty 50 Stock Analysis: Comprehensive Research Notebook
This notebook contains 5 segments of analysis as per the project guidelines.

### Step 1: Setup and Libraries

In [ ]:
!pip install PyYAML
import yaml
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

### Step 2: Data Extraction (YAML to CSV)
Convert date-wise YAML files into 50 symbol-wise CSV files.

In [ ]:
# NOTE: In Colab, make sure you upload the 'Test data' folder to /content/
RAW_DATA_PATH = '/content/Test data/original_yamls' 
PROCESSED_PATH = '/content/Test data/nifty50_data'
os.makedirs(PROCESSED_PATH, exist_ok=True)

all_dfs = []
path_month = [
    "2023-10", "2023-11", "2023-12", 
    "2024-01", "2024-02", "2024-03", 
    "2024-04", "2024-05", "2024-06", 
    "2024-07", "2024-08", "2024-09", 
    "2024-10", "2024-11"
]

for month in path_month:
    month_path = os.path.join(RAW_DATA_PATH, month)
    path_date = glob(os.path.join(month_path, "*.yaml"))
    for file_path in path_date:
        with open(file_path, 'r') as file:
            data = yaml.safe_load(file)
            all_dfs.append(pd.DataFrame(data))

wholedf = pd.concat(all_dfs, ignore_index=True)
cols = ['date', 'open', 'high', 'low', 'close', 'volume', 'Ticker']
wholedf = wholedf[cols]

for t in wholedf['Ticker'].unique():
    df_t = wholedf[wholedf['Ticker'] == t].sort_values(by='date')
    df_t.to_csv(f'{PROCESSED_PATH}/{t}.csv', index=False)

print(f"Generated 50 CSV files in {PROCESSED_PATH}")

### Segment 1: Volatility Analysis
Visualizing the standard deviation of daily returns.

In [ ]:
vol_list = []
csv_files = glob(f'{PROCESSED_PATH}/*.csv')
for fp in csv_files:
    df = pd.read_csv(fp)
    t = os.path.basename(fp).replace('.csv', '')
    df['ret'] = df['close'].pct_change()
    vol_list.append({'Ticker': t, 'Volatility': df['ret'].std()})

vol_df = pd.DataFrame(vol_list).sort_values(by='Volatility', ascending=False)
display(vol_df.head(10))

### Segment 2: Cumulative Return Over Time
Growth of the Top 5 performing stocks.

In [ ]:
top_5 = []
for fp in csv_files:
    df = pd.read_csv(fp)
    t = os.path.basename(fp).replace('.csv', '')
    ret = (df['close'].iloc[-1] - df['close'].iloc[0]) / df['close'].iloc[0]
    top_5.append({'Ticker': t, 'Return': ret})

top_5_tickers = pd.DataFrame(top_5).sort_values(by='Return', ascending=False).head(5)['Ticker'].tolist()

plt.figure(figsize=(10, 5))
for t in top_5_tickers:
    df = pd.read_csv(f'{PROCESSED_PATH}/{t}.csv')
    df['cum_ret'] = (1 + df['close'].pct_change().fillna(0)).cumprod()
    plt.plot(df['date'], df['cum_ret'], label=t)

plt.legend(); plt.title('Top 5 Cumulative Returns'); plt.xticks(rotation=45); plt.show()

### Segment 3: Sector-wise Performance
Analyzing returns based on industry sectors using the provided CSV.

In [ ]:
sector_data = pd.read_csv('/content/Test data/Sector_data.csv')
sector_data['Ticker'] = sector_data['Symbol'].str.split(': ').str[-1].str.strip()

sector_perf = []
for index, row in sector_data.iterrows():
    t, s = row['Ticker'], row['sector']
    try:
        df = pd.read_csv(f'{PROCESSED_PATH}/{t}.csv')
        ret = (df['close'].iloc[-1] - df['close'].iloc[0]) / df['close'].iloc[0]
        sector_perf.append({'Sector': s, 'Return': ret})
    except: continue

sec_df = pd.DataFrame(sector_perf).groupby('Sector').mean().sort_values(by='Return', ascending=False)
display(sec_df)

### Segment 4: Stock Price Correlation
Heatmap showing how stocks move together.

In [ ]:
prices = {}
for t in top_5_tickers:
    df = pd.read_csv(f'{PROCESSED_PATH}/{t}.csv')
    prices[t] = df['close']
sns.heatmap(pd.DataFrame(prices).corr(), annot=True, cmap='RdYlGn')
plt.title('Correlation of Top 5 Stocks'); plt.show()

### Segment 5: Top 5 gainers & losers ( Month wise )
Granular monthly performance breakdown.

In [ ]:
wholedf['date'] = pd.to_datetime(wholedf['date'])
wholedf['month'] = wholedf['date'].dt.strftime('%Y-%m')
monthly = wholedf.groupby(['month', 'Ticker'])['close'].agg(['first', 'last'])
monthly['ret'] = (monthly['last'] - monthly['first']) / monthly['first']

for m in wholedf['month'].unique()[:3]:
    print(f"\n--- {m} Top 5 Gainers ---")
    display(monthly.loc[m].sort_values(by='ret', ascending=False).head(5))
    print(f"\n--- {m} Top 5 Losers ---")
    display(monthly.loc[m].sort_values(by='ret', ascending=True).head(5))